In [1]:
# Cell 1 -- check what GPU you got
!nvidia-smi

Mon Sep  7 09:58:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 -- clone and install
!git clone https://github.com/swapnilanilkale/panorama-onco.git
%cd panorama-onco
# Colab runs Python 3.13; ADR-0004 pins <3.13 for local ecosystem reasons that
# do not apply here, so bypass the check rather than change the pin.
%pip install -q --ignore-requires-python -e ".[dev]"

Cloning into 'panorama-onco'...
remote: Enumerating objects: 524, done.
remote: Counting objects: 100% (370/370), done.
remote: Compressing objects: 100% (223/223), done.
remote: Total 524 (delta 176), reused 303 (delta 118), pack-reused 154 (from 1)
Receiving objects: 100% (524/524), 209.00 KiB | 4.10 MiB/s, done.
Resolving deltas: 100% (239/239), done.
/content/panorama-onco
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/98

In [3]:
# Cell 3 Verifying Working Directory

import sys, os
print("cwd:", os.getcwd())
print("src exists:", os.path.exists('/content/panorama-onco/src/panorama'))
!pip show panorama-onco 2>/dev/null | head -3

# The editable install writes a path file that the ALREADY-RUNNING kernel has
# not read. Point at src/ directly -- simpler than restarting the runtime.
if '/content/panorama-onco/src' not in sys.path:
    sys.path.insert(0, '/content/panorama-onco/src')

import panorama
from panorama.vision.encoder import MultiStreamViT
from panorama.data.dicom import read_series
print("panorama imports OK")

cwd: /content/panorama-onco
src exists: True
Name: panorama-onco
Version: 0.0.1
Summary: 
panorama imports OK


In [ ]:
# Cell 3 -- regenerate the data (ADR-0006: the recipe is versioned, not the data)
!python scripts/build_synthetic_cohort.py --patients 200 --max-studies 4
!python scripts/build_report_corpus.py

In [6]:
%cd /content/panorama-onco
!git pull

/content/panorama-onco
Already up to date.


In [7]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/panorama-data
!cp -r data/tcia/qin-breast-nifti /content/drive/MyDrive/panorama-data/
!cp -r data/tcia/manifests /content/drive/MyDrive/panorama-data/
!du -sh /content/drive/MyDrive/panorama-data/*

Mounted at /content/drive
cp: cannot stat 'data/tcia/qin-breast-nifti': No such file or directory
cp: cannot stat 'data/tcia/manifests': No such file or directory
26K	/content/drive/MyDrive/panorama-data/manifests
2.8G	/content/drive/MyDrive/panorama-data/qin-breast-nifti


In [8]:
import os, shutil
if os.path.exists('/content/drive/MyDrive/panorama-data/qin-breast-nifti'):
    os.makedirs('data/tcia', exist_ok=True)
    shutil.copytree('/content/drive/MyDrive/panorama-data/qin-breast-nifti',
                    'data/tcia/qin-breast-nifti', dirs_exist_ok=True)
    shutil.copytree('/content/drive/MyDrive/panorama-data/manifests',
                    'data/tcia/manifests', dirs_exist_ok=True)
    print("restored from Drive")
else:
    print("no cached data -- run download and convert")

restored from Drive


In [13]:
!python scripts/precompute_volumes.py 2>&1 | tail -3

2026-09-05 08:53:47 | INFO     | __main__ |   90/105 studies
2026-09-05 08:54:37 | INFO     | __main__ |   100/105 studies
2026-09-05 08:55:00 | INFO     | __main__ | done: 420 arrays, 7.0 GB


In [14]:
import os
print("DICOM downloaded:", os.path.exists('data/tcia/qin-breast'))
print("NIfTI converted :", os.path.exists('data/tcia/qin-breast-nifti'))
!du -sh data/tcia/* 2>/dev/null
!ls data/tcia/

DICOM downloaded: False
NIfTI converted : True
28K	data/tcia/manifests
2.8G	data/tcia/qin-breast-nifti
6.6G	data/tcia/qin-breast-preprocessed
manifests  qin-breast-nifti  qin-breast-preprocessed


In [15]:
# build the manifest  <-- the new cell
from panorama.data.manifest import scan_directory, write_manifest
from panorama.core.logging import configure_logging

configure_logging("INFO")
studies = scan_directory("data/tcia/qin-breast-nifti")
write_manifest(studies, "data/tcia/manifests/qin-breast.csv")
print(f"{len(studies)} studies, {len({s.patient_id for s in studies})} patients")

105 studies, 38 patients


In [8]:
import os
os.chdir('/content/panorama-onco')

from panorama.train.pretrain import main

main([
    "configs/pretrain_qin.yaml",
    "data.precomputed_root=data/tcia/qin-breast-preprocessed",
    "output_dir=/content/drive/MyDrive/panorama-runs/qin-large",
    "model.embed_dim=768", "model.depth=12", "model.num_heads=12",
    "model.decoder_dim=512", "model.decoder_depth=4",
    "data.batch_size=8", "data.num_workers=2",
    "model.max_steps=3000", "trainer.max_steps=3000", "model.warmup_steps=200",
    "trainer.accelerator=gpu", "trainer.precision=16-mixed",
])

ValueError: module functions cannot set METH_CLASS or METH_STATIC

In [11]:
import os
os.chdir('/content/panorama-onco')

from panorama.train.pretrain import main

main([
    "configs/pretrain_qin.yaml",
    "data.precomputed_root=data/tcia/qin-breast-preprocessed",
    "model.embed_dim=768", "model.depth=12", "model.num_heads=12",
    "model.decoder_dim=512", "model.decoder_depth=4",
    "data.batch_size=8", "data.num_workers=2",
    "model.max_steps=20000", "trainer.max_steps=20000", "model.warmup_steps=500",
    "trainer.accelerator=gpu", "trainer.precision=16-mixed",
])

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_R

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ MultiModalMAE │  184 M │ train │     0 │
└───┴───────┴───────────────┴────────┴───────┴───────┘

Trainable params: 184 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 184 M                                                                                                
Total estimated model params size (MB): 738.523                                                                    
Modules in train mode: 351                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

: 

In [6]:
import csv
from pathlib import Path
v = max(Path("outputs/qin").glob("*/lightning_logs/version_*"),
        key=lambda p: p.stat().st_mtime)
rows = [r for r in csv.DictReader((v/"metrics.csv").open()) if r.get("val/loss")]
for r in rows[-8:]:
    print(f"step {r['step']:>6}  val_loss {float(r['val/loss']):.4f}  "
          f"var_expl {1-float(r['val/loss']):+.4f}  "
          f"rank {r.get('val/effective_rank','-')}")

ValueError: max() iterable argument is empty

In [2]:
from pathlib import Path
print("run dirs:", [p.name for p in Path("outputs/qin").glob("*/")])

run dirs: []


In [4]:
# Cell 5 -- the experiment ADR-0009 points to: does capacity fix under-learning?
import os
os.chdir('/content/panorama-onco')

from panorama.train.pretrain import main

main([
    "configs/pretrain_qin.yaml",
    "trainer.accelerator=gpu",
    "trainer.precision=16-mixed",
    "data.num_workers=2",
    "data.batch_size=8",
])

KeyboardInterrupt: 

In [5]:
# Cell 6 -- save results back, since the VM is wiped on disconnect
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/panorama-runs
!cp -r outputs/qin /content/drive/MyDrive/panorama-runs/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat 'outputs/qin': No such file or directory
